# Probabilmente, approssimativamente corretto

Il codice del capitolo [«Probabilmente, approssimativamente corretto»](https://book.paithon.it/main/TeoriaApprendimento/pac.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy scikit-learn

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Probabilmente, approssimativamente corretto

[Leggi la pagina](https://book.paithon.it/main/TeoriaApprendimento/pac.html)


### Contare i sospettati


In [ ]:
import numpy as np
from math import ceil, log

rng = np.random.default_rng(0)
N_IPOTESI = 1000                         # soglie t = 0,000; 0,001; ...; 0,999
VERA = 0.371                             # la soglia che genera le etichette
EPS, DELTA = 0.05, 0.01

# gli esempi che chiede il conto dei sospettati, nelle due forme
m_esatto = ceil(log(DELTA / N_IPOTESI) / log(1 - EPS))
m_bound = ceil((log(N_IPOTESI) + log(1 / DELTA)) / EPS)
print(f"conto con (1-eps)^m: {m_esatto} esempi; con e^(-eps m): {m_bound}")
print(f"con un milione di ipotesi: {ceil(log(DELTA / 1e6) / log(1 - EPS))} esempi")
print(f"con eps = 0,01: {ceil(log(DELTA / N_IPOTESI) / log(1 - 0.01))} esempi")

def quota_di_fallimenti(m, prove=20_000):
    """Su quanti campioni di m esempi sopravvive una soglia compatibile con
    tutti gli esempi ma con errore vero oltre EPS (l'errore di t è |t - VERA|)."""
    x = rng.random((prove, m))
    positivo = x >= VERA
    ultimo_neg = np.where(~positivo, x, -1.0).max(axis=1)
    primo_pos = np.where(positivo, x, 2.0).min(axis=1)
    griglia = np.arange(N_IPOTESI) / N_IPOTESI
    # le soglie compatibili sono quelle in (ultimo_neg, primo_pos]
    piu_bassa = np.searchsorted(griglia, ultimo_neg, side="right")
    piu_alta = np.searchsorted(griglia, primo_pos, side="right") - 1
    peggiore = np.maximum(VERA - griglia[piu_bassa], griglia[piu_alta] - VERA)
    return (peggiore > EPS + 1e-12).mean()

print(f"con {m_bound} esempi sopravvive una soglia cattiva nel "
      f"{quota_di_fallimenti(m_bound):.2%} dei campioni")
for m in (40, 80, 100, 120):
    print(f"  con {m:>3} esempi: {quota_di_fallimenti(m):.2%}")

# il conto esatto: nessun esempio nei due tratti larghi EPS + 1/N_IPOTESI
def p_fallimento(m, largo=EPS + 1 / N_IPOTESI):
    return 2 * (1 - largo) ** m - (1 - 2 * largo) ** m

for m in (80, 100, 102, 120):
    print(f"  esatto, {m:>3} esempi: {p_fallimento(m):.2%}")

# senza la regola giusta nella lista: la stima di ogni errore deve stare entro 0,005
dev = 0.005
print(f"caso agnostico, deviazione {dev}: "
      f"{ceil((log(N_IPOTESI) + log(2 / DELTA)) / (2 * dev**2))} esempi")

## Quando le ipotesi sono infinite: la dimensione VC

[Leggi la pagina](https://book.paithon.it/main/TeoriaApprendimento/dimensione-vc.html)


### Dal caso peggiore a un polinomio


In [ ]:
import numpy as np
from itertools import combinations
from math import comb, e, log, sqrt

rng = np.random.default_rng(1)

def colorazioni_di_rette(punti):
    """Le colorazioni che una retta produce sui punti, cercate tutte.
    Ogni colorazione realizzabile si ottiene, spostando appena la retta, da
    una retta che passa per due dei punti: si decide poi da che parte stanno
    quei due con una rotazione o una traslazione minuscola."""
    m = len(punti)
    trovate = {tuple([1] * m), tuple([0] * m)}
    for i, j in combinations(range(m), 2):
        d = punti[j] - punti[i]
        lato = np.sign((punti - punti[i]) @ np.array([-d[1], d[0]]))
        for a in (1, -1):
            for b in (1, -1):
                y = lato.copy()
                y[i], y[j] = a, b
                for verso in (1, -1):                    # e la retta capovolta
                    trovate.add(tuple(int(v > 0) for v in verso * y))
    return trovate

for m in (3, 4, 5, 10, 20):
    n = len(colorazioni_di_rette(rng.random((m, 2))))
    sauer = sum(comb(m, i) for i in range(4))           # dimensione VC d = 3
    print(f"m={m:>2}: tutte {2**m:>7}, con una retta {n:>3}"
          f" (m^2-m+2 = {m * m - m + 2}), tetto di Sauer {sauer:>4}")

# il bound di Mohri, corollario 3.19, per le rette del piano (d = 3), delta = 0,01
d, delta = 3, 0.01
for m in (100, 1_000, 10_000, 100_000):
    scarto = sqrt(2 * d * log(e * m / d) / m) + sqrt(log(1 / delta) / (2 * m))
    print(f"m={m:>6}: rischio vero al massimo rischio empirico + {scarto:.3f}")

## Una complessità misurata sui dati: Rademacher e il margine

[Leggi la pagina](https://book.paithon.it/main/TeoriaApprendimento/rademacher-margine.html)


### La strada larga


In [ ]:
import numpy as np
from math import e, log, sqrt

rng = np.random.default_rng(2)

def rademacher_lineare(X, norma_w=1.0, estrazioni=2000):
    """Complessità di Rademacher empirica di {x -> w.x : ||w|| <= norma_w}.
    Il massimo su w si fa a mano: per Cauchy-Schwarz vale
    norma_w / m * ||somma_i sigma_i x_i||, e resta da mediare sulle monete."""
    m = len(X)
    sigma = rng.choice([-1.0, 1.0], size=(estrazioni, m))
    return norma_w / m * np.linalg.norm(sigma @ X, axis=1).mean()

def sulla_sfera(m, d):
    X = rng.standard_normal((m, d))
    return X / np.linalg.norm(X, axis=1, keepdims=True)   # tutti con ||x|| = 1

RHO = 0.5                                    # il margine richiesto
for d in (2, 1000):
    for m in (10, 100, 1000):
        rad = rademacher_lineare(sulla_sfera(m, d))
        # i due termini: margine (2/rho) e VC (iperpiani con termine noto: d + 1)
        margine = 2 / RHO * rad
        vc = sqrt(2 * (d + 1) * log(e * m / (d + 1)) / m) if m > d + 1 else float("inf")
        print(f"d={d:>4} m={m:>4}: Rademacher {rad:.4f} (tetto {1 / sqrt(m):.4f});"
              f" termine di margine {margine:.3f}, termine VC {vc:.3f}")

## Quello che le garanzie non spiegano

[Leggi la pagina](https://book.paithon.it/main/TeoriaApprendimento/garanzie-e-reti.html)


### Uno studente che impara qualunque cosa


In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X, y = load_digits(return_X_y=True)                  # cifre scritte a mano, 8x8
X = X / 16.0
X_tr, X_te, y_tr, y_te = train_test_split(X, y, train_size=300, random_state=0)
rng = np.random.default_rng(0)

def addestra_finche_impara(etichette, classi, tetto=2000):
    """Epoche su epoche finché la rete non indovina tutti gli esempi."""
    rete = MLPClassifier(hidden_layer_sizes=(512,), alpha=0.0,
                         learning_rate_init=0.01, batch_size=50, random_state=0)
    for epoca in range(1, tetto + 1):
        rete.partial_fit(X_tr, etichette, classes=classi)
        if rete.score(X_tr, etichette) == 1.0:
            break
    return rete, epoca

rimescolate = rng.permutation(y_tr)                  # le stesse etichette, in disordine
sbagliate = (y_tr + rng.integers(1, 10, size=len(y_tr))) % 10   # mai quella giusta
print(f"etichette rimaste giuste nel rimescolamento: {(rimescolate == y_tr).mean():.1%}")
for nome, etichette in (("vere", y_tr), ("rimescolate", rimescolate),
                        ("sempre sbagliate", sbagliate)):
    rete, epoche = addestra_finche_impara(etichette, np.arange(10))
    print(f"etichette {nome:>16}: addestramento {rete.score(X_tr, etichette):.1%},"
          f" prova {rete.score(X_te, y_te):.1%}, {epoche} epoche")

# la complessità di Rademacher della rete su questi 300 esempi: segni tirati a sorte
correlazioni = []
for _ in range(3):
    sigma = rng.choice([-1, 1], size=len(y_tr))
    rete, _ = addestra_finche_impara(sigma, np.array([-1, 1]))
    correlazioni.append(np.mean(sigma * rete.predict(X_tr)))
print(f"correlazione con i segni casuali, tre sorteggi: {np.round(correlazioni, 2)}")